# 02 · Train Encoders — Multi-model comparison

This notebook is the **orchestrator**: it imports all business logic from `src/`
and loops over models × augmentation strategies × loss functions.

All results are saved to `results/` as JSON.

---
### Quick-start
1. Adjust `MODELS_TO_RUN`, `AUGMENTATIONS`, `LOSSES` in the **Configuration** cell.
2. Run all cells top to bottom.
3. Results appear in `results/` and in the final comparison table.

## 0 · Setup (Colab only)

In [1]:
# ── Google Colab setup ────────────────────────────────────────────────────────
# This cell is a no-op when running locally.
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Mount Drive to persist results across sessions (optional)
    from google.colab import drive
    drive.mount('/content/drive')

    # Clone the project repo if running directly from Colab
    # !git clone https://github.com/YOUR_USER/YOUR_REPO.git
    # %cd YOUR_REPO

    # Install dependencies
    !pip install -q transformers datasets scikit-learn torch nltk rich
    !pip uninstall -y torchvision # Avoid potential version conflicts with Colab's pre-installed version

    print('Colab dependencies installed.')
else:
    print('Running locally — skipping Colab setup.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab dependencies installed.


In [2]:
# Add project root to sys.path so `src/` is importable from any working directory
import sys
from pathlib import Path

# Move the project to `progettoLLM/` folder in your Drive and update the path below if needed
PROJECT_ROOT = Path('/content/drive/MyDrive/progettoLLM/CLARITY').resolve()   # notebooks/ is one level below root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)

Project root: /content/drive/MyDrive/progettoLLM/CLARITY


## 1 · Configuration

In [10]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# EDIT THIS CELL to control what gets trained.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from config.model_configs import MODEL_CONFIGS
from src.data.augmentation import list_augmentations

# ── Models to benchmark ───────────────────────────────────────────────────────
# Comment out any model you don't want to run.
MODELS= [
    'distilbert-base-uncased',
    #'longformer-base',              # original notebook baseline
    'roberta-base',                 # paper baseline
    # 'xlnet-base',                 # paper baseline — slower, enable if needed
    # 'longformer-large',           # needs >20 GB VRAM
    #'bert-base-uncased',          # reference

]

# ── Balancing strategies ───────────────────────────────────────────────────────
# Available: 'none', 'semantic_downsampling', 'Paraphrase Upsampling', 'Smart Resampling'
# Note: it may takes a while, especially upsampling strategy
BALANCING_STRATEGIES = [
    'none',             # control: no balancing
    #'semantic_downsampling',  # downsample majority classes based on semantic similarity
    # 'paraphrase_upsampling',  # augment minority classes with paraphrases
    #'smart_resampling',  # combine downsampling and upsampling based on class difficulty
]

# ── Augmentation strategies ───────────────────────────────────────────────────
# Available: print(list_augmentations())
AUGMENTATIONS = [
    'none',                  # control: no augmentation
    # 'synonym_replacement',   # EDA-style synonym swap on minority classes
    # 'oversampling',          # duplicate minority examples to balance
    # 'random_deletion',
    # 'random_swap',
    # 'back_translation',     # slow — needs GPU and internet
    # 'length_category',      # adds a length-based feature (ablation)
    #'tone_analysis',          # adds a tone-based feature (ablation)
]

# ── Loss functions ────────────────────────────────────────────────────────────
# Available: 'focal', 'weighted_ce', 'label_smoothing', 'ce'
LOSSES = [
    #'focal',         # original notebook loss
    # 'weighted_ce', # ablation: class-weighted CE without focal modulation
    'ce',          # ablation: vanilla cross-entropy (no weighting)
    # 'label_smoothing',  # ablation: CE with label smoothing instead of class weights
]

# ── Shared hyperparameters ────────────────────────────────────────────────────
NUM_EPOCHS    = 5
WEIGHT_DECAY  = 0.01
FOCAL_GAMMA   = 2.0
SEED          = 42

# ── Task setting ──────────────────────────────────────────────────────────────
# 'evasion'  → 9-class fine-grained task  (used in the original notebook)
# 'clarity'  → 3-class high-level task    (Clear Reply / Ambivalent / Clear Non-Reply)
TASK = 'evasion'

print('Models     :', MODELS)
print('Balancing  :', BALANCING_STRATEGIES)
print('Augment.   :', AUGMENTATIONS)
print('Losses     :', LOSSES)
print('Total runs :', len(MODELS) * len(AUGMENTATIONS) * len(LOSSES))

Models     : ['distilbert-base-uncased', 'roberta-base']
Balancing  : ['none']
Augment.   : ['none']
Losses     : ['ce']
Total runs : 2


## 2 · Load dataset (once — shared by all runs)

In [11]:
from src.data.dataset_loader import load_and_split_dataset, EVASION_TO_CLARITY

print('Loading ailsntua/QEvasion ...')
raw_ds = load_and_split_dataset(seed=SEED, verbose=True)
raw_ds

Loading ailsntua/QEvasion ...
  train       :  2758 samples
  validation  :   690 samples
  test        :   308 samples


DatasetDict({
    train: Dataset({
        features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label'],
        num_rows: 2758
    })
    validation: Dataset({
        features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label'],
        num_rows: 690
    })
    test: Dataset({
        features: ['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'anno

## 3 · Build label maps (once)

In [12]:
from src.data.label_utils import build_label_maps, compute_alpha_weights, apply_labels

# Select the label column based on the task
import src.data.label_utils as lu
if TASK == 'clarity':
    lu.LABEL_COLUMN = 'clarity_label'   # switch to 3-class task
else:
    lu.LABEL_COLUMN = 'evasion_label'   # default 9-class task

label2id, id2label = build_label_maps(raw_ds)
num_classes = len(label2id)

print(f'Task          : {TASK}')
print(f'Num classes   : {num_classes}')
print(f'Label mapping : {label2id}')

# Apply integer labels to all splits
labeled_ds = apply_labels(raw_ds, label2id, verbose=True)

Task          : evasion
Num classes   : 9
Label mapping : {'Claims ignorance': 0, 'Clarification': 1, 'Declining to answer': 2, 'Deflection': 3, 'Dodging': 4, 'Explicit': 5, 'General': 6, 'Implicit': 7, 'Partial/half-answer': 8}


Map:   0%|          | 0/308 [00:00<?, ? examples/s]


[train] label distribution:
  Claims ignorance          :   98
  Clarification             :   72
  Declining to answer       :  118
  Deflection                :  307
  Dodging                   :  557
  Explicit                  :  859
  General                   :  310
  Implicit                  :  371
  Partial/half-answer       :   66

[validation] label distribution:
  Claims ignorance          :   21
  Clarification             :   20
  Declining to answer       :   27
  Deflection                :   74
  Dodging                   :  149
  Explicit                  :  193
  General                   :   76
  Implicit                  :  117
  Partial/half-answer       :   13

[test] label distribution:
  Claims ignorance          :    8
  Clarification             :    4
  Declining to answer       :   11
  Deflection                :   20
  Dodging                   :   50
  Explicit                  :   79
  General                   :   65
  Implicit                  :   67

## 4 · Main training loop

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, EarlyStoppingCallback

from config.model_configs import get_model_config
from src.data.augmentation import get_augmentation_fn
from src.training.metrics import build_compute_metrics_fn, compute_detailed_report
from src.training.trainers import get_trainer
from src.utils.env_utils import get_training_args
from src.utils.results_utils import save_results


def free_memory():
    """Release GPU/CPU memory between runs."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def tokenize_for_model(ds, tokenizer, model_cfg: dict):
    """Tokenize the dataset for a specific model configuration."""
    max_length = model_cfg['max_length']
    use_token_type_ids = model_cfg.get('token_type_ids', False)

    def tokenize_fn(examples):
        modified_answers = []

        # Verifichiamo se le feature esistono in questo split del dataset
        has_tone = 'tone' in examples
        has_length = 'length_category' in examples


        for i in range(len(examples['interview_answer'])):
            ans = str(examples['interview_answer'][i])

             # Estraiamo i valori solo se la colonna esiste, altrimenti stringa vuota
            t = str(examples['tone'][i]) if has_tone and examples['tone'][i] else ""
            l = str(examples['length_category'][i]) if has_length and examples['length_category'][i] else ""

            prefix_parts = []
            if t:
                prefix_parts.append(f"Tone: {t}")
            if l:
                prefix_parts.append(f"Length: {l}")

            if prefix_parts:
                prefix = " | ".join(prefix_parts) + " | Answer: "
            else:
                prefix = ""

            modified_answers.append(prefix + ans)

        return tokenizer(
            examples['question'],
            modified_answers, # Passiamo le risposte modificate col prefisso
            padding='max_length',
            truncation=True,
            max_length=max_length,
        )

    tokenized = ds.map(tokenize_fn, batched=True)

    # Keep only the columns the model actually needs
    columns_to_keep = ['input_ids', 'attention_mask', 'label']
    if use_token_type_ids:
        columns_to_keep.append('token_type_ids')

    # Some models (e.g. Longformer) also produce global_attention_mask
    if 'global_attention_mask' in tokenized['train'].column_names:
        columns_to_keep.append('global_attention_mask')

    tokenized.set_format(type='torch', columns=columns_to_keep)
    return tokenized


aug_tag_name = "_".join(AUGMENTATIONS)
bal_tag_name = "_".join(BALANCING_STRATEGIES)

# ── Outer loop ────────────────────────────────────────────────────────────────
all_run_results = []
master_ds = labeled_ds
for bal in BALANCING_STRATEGIES:
    print(f"Applying Resampling: {bal}")
    master_ds = get_augmentation_fn(bal)(master_ds, label2id, seed=SEED)
    free_memory()

# B. NLP Augmentations
for aug in AUGMENTATIONS:
    print(f"Applying Augmentation: {aug}")
    master_ds = get_augmentation_fn(aug)(master_ds, label2id, seed=SEED)
    free_memory()

print(f"\nFinal Train Size: {len(master_ds['train'])}")

# Compute inverse-frequency weights (for FocalLoss / WeightedCE)
alpha_tensor = compute_alpha_weights(master_ds, label2id)
print('\nAlpha weights (inverse frequency):')
for i, w in enumerate(alpha_tensor.tolist()):
    print(f'  {id2label[i]:<25} : {w:.4f}')

for model_key in MODELS:
    model_cfg = get_model_config(model_key)
    model_id  = model_cfg['model_id']

    print(f'\n{"="*70}')
    print(f'MODEL : {model_key}  ({model_id})')
    print(f'{"="*70}')

    # Load tokenizer once per model
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenized_ds = tokenize_for_model(master_ds, tokenizer, model_cfg)


    for loss_name in LOSSES:
        print(f'\n    ── Loss: {loss_name} ──')

        run_tag = f'{model_key} / {loss_name}'

        # ── Build model (fresh weights each run) ──────────────────────────
        model = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            num_labels=num_classes,
            id2label=id2label,
            label2id=label2id,
            ignore_mismatched_sizes=True,
        )

        # ── Build TrainingArguments ───────────────────────────────────────
        training_args = get_training_args(
            model_key=model_key,
            model_cfg=model_cfg,
            bal_name=bal_tag_name,
            aug_name=aug_tag_name,
            loss_name=loss_name,
            num_epochs=NUM_EPOCHS,
            weight_decay=WEIGHT_DECAY,
            task=TASK
        )

        # Configure dynamic metrics computing depending on task
        eval_mapping = EVASION_TO_CLARITY if TASK == 'evasion' else None
        custom_compute_metrics = build_compute_metrics_fn(
            id2label=id2label,
            evasion_to_clarity=eval_mapping
        )

        # ── Build Trainer ─────────────────────────────────────────────────
        trainer = get_trainer(
            loss_name=loss_name,
            model=model,
            training_args=training_args,
            train_dataset=tokenized_ds['train'],
            eval_dataset=tokenized_ds['validation'],
            compute_metrics=custom_compute_metrics,
            alpha=alpha_tensor,
            gamma=FOCAL_GAMMA,
            num_classes=num_classes,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
        )

        # ── Train ─────────────────────────────────────────────────────────
        print(f'    Starting training for: {run_tag}')
        trainer.train()

        # ── Evaluate on VALIDATION set ────────────────────────────────────
        val_metrics = trainer.evaluate()
        print(f'    Validation metrics: {val_metrics}')

        # ── Evaluate on TEST set ──────────────────────────────────────────
        test_pred = trainer.predict(tokenized_ds['test'])
        test_metrics = test_pred.metrics
        print(f'    Test metrics: {test_metrics}')

        # Detailed per-class report
        report = compute_detailed_report(
            test_pred.predictions,
            test_pred.label_ids,
            id2label,
            evasion_to_clarity=eval_mapping,
        )
        print(f'\n    Per-class report:\n{report}')

        # ── Save results ──────────────────────────────────────────────────
        metrics_dict = {
            'val_macro_f1':      val_metrics.get('eval_macro_f1', 0),
            'val_weighted_f1':   val_metrics.get('eval_weighted_f1', 0),
            'val_accuracy':      val_metrics.get('eval_accuracy', 0),
            'test_macro_f1':     test_metrics.get('test_macro_f1', 0),
            'test_weighted_f1':  test_metrics.get('test_weighted_f1', 0),
            'test_accuracy':     test_metrics.get('test_accuracy', 0),
        }

        if eval_mapping:
            metrics_dict.update({
                'val_clarity_macro_f1':      val_metrics.get('eval_clarity_macro_f1', 0),
                'val_clarity_accuracy':      val_metrics.get('eval_clarity_accuracy', 0),
                'test_clarity_macro_f1':     test_metrics.get('test_clarity_macro_f1', 0),
                'test_clarity_accuracy':     test_metrics.get('test_clarity_accuracy', 0),
            })

        save_results(
            model_key=model_key,
            bal_name=bal_tag_name,
            aug_name=aug_tag_name,
            loss_name=loss_name,
            metrics=metrics_dict,
            extra={
                'model_id':    model_id,
                'max_length':  model_cfg['max_length'],
                'num_epochs':  NUM_EPOCHS,
                'focal_gamma': FOCAL_GAMMA,
                'task':        TASK,
                'train_size':  len(master_ds['train']),
            },
        )
        all_run_results.append(run_tag)

        # ── Clean up before next run ──────────────────────────────────────
        del model, trainer
        free_memory()

    del tokenized_ds, tokenizer
    free_memory()

print('\n\nAll runs completed:')
for r in all_run_results:
    print(' ✓', r)

Applying Resampling: none
Applying Augmentation: none

Final Train Size: 2758

Alpha weights (inverse frequency):
  Claims ignorance          : 3.1270
  Clarification             : 4.2562
  Declining to answer       : 2.5970
  Deflection                : 0.9982
  Dodging                   : 0.5502
  Explicit                  : 0.3567
  General                   : 0.9885
  Implicit                  : 0.8260
  Partial/half-answer       : 4.6431

MODEL : distilbert-base-uncased  (distilbert-base-uncased)


Map:   0%|          | 0/2758 [00:00<?, ? examples/s]

Map:   0%|          | 0/308 [00:00<?, ? examples/s]


    ── Loss: ce ──


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Macro F1,Weighted F1,Accuracy,Clarity Macro F1,Clarity Weighted F1,Clarity Accuracy
1.237383,1.623874,5,0.367178,0.408521,0.437681,0.612398,0.667170,0.660870


    Validation metrics: {'eval_loss': 1.6238744258880615, 'eval_macro_f1': 0.36717769280777823, 'eval_weighted_f1': 0.40852065098355617, 'eval_accuracy': 0.43768115942028984, 'eval_clarity_macro_f1': 0.6123976324397252, 'eval_clarity_weighted_f1': 0.6671700686897449, 'eval_clarity_accuracy': 0.6608695652173913}


    Test metrics: {'test_loss': 1.764695167541504, 'test_macro_f1': 0.2328825638903158, 'test_weighted_f1': 0.2578637602874993, 'test_accuracy': 0.30844155844155846, 'test_clarity_macro_f1': 0.49620051526234676, 'test_clarity_weighted_f1': 0.565686914269224, 'test_clarity_accuracy': 0.5487012987012987, 'test_runtime': 2.3408, 'test_samples_per_second': 131.581, 'test_steps_per_second': 4.272}

    Per-class report:
                     precision    recall  f1-score   support

   Claims ignorance       0.36      0.62      0.45         8
      Clarification       0.50      0.25      0.33         4
Declining to answer       0.16      0.36      0.22        11
         Deflection       0.05      0.05      0.05        20
            Dodging       0.23      0.32      0.27        50
           Explicit       0.40      0.68      0.50        79
            General       0.00      0.00      0.00        65
           Implicit       0.38      0.21      0.27        67
Partial/half-answer       0.00 

## 5 · Quick comparison table

In [3]:
from src.utils.results_utils import compare_results, print_comparison_table

df = compare_results(
    metrics_to_show=['test_macro_f1', 'test_weighted_f1', 'test_accuracy',
                     'val_macro_f1']
)
df

,model,task,balancing,augmentation,loss,timestamp,test_macro_f1,test_weighted_f1,test_accuracy,val_macro_f1
0,distilbert-base-uncased,clarity,none,none,ce,2026-06-19T10:35:32,0.5454,0.6358,0.6429,0.6067
1,distilbert-base-uncased,clarity,smart_resampling,tone_analysis,ce,2026-05-29T09:05:14,0.5101,0.6011,0.6006,0.5482
2,roberta-base,clarity,none,none,ce,2026-06-19T10:45:16,0.5581,0.6430,0.6396,0.6028
3,roberta-base,clarity,smart_resampling,tone_analysis,ce,2026-06-19T11:46:35,0.5504,0.6452,0.6494,0.6114
4,distilbert-base-uncased,evasion,none,none,ce,2026-06-20T15:14:35,0.2114,0.1927,0.2597,0.3044
5,distilbert-base-uncased,evasion,smart_resampling,tone_analysis,ce,2026-06-20T14:54:58,0.2918,0.2171,0.2370,0.3446
6,roberta-base,evasion,none,none,ce,2026-06-20T15:24:23,0.2329,0.2579,0.3084,0.3672
7,roberta-base,evasion,smart_resampling,tone_analysis,ce,2026-06-20T15:09:00,0.3078,0.2852,0.3084,0.3616


In [4]:
# Rich table (if rich is installed, otherwise plain print)
print_comparison_table()

                                  Experiment Results (sorted by Test Macro F1 ↓)                                   
┏━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃             ┃         ┃             ┃ augmentatio ┃      ┃ test_macro_ ┃ test_weight ┃ test_accur ┃ val_macro_f ┃
┃ model       ┃ task    ┃ balancing   ┃ n           ┃ loss ┃          f1 ┃       ed_f1 ┃        acy ┃           1 ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ roberta-bas │ clarity │ none        │ none        │ ce   │      0.5581 │      0.6430 │     0.6396 │      0.6028 │
│ e           │         │             │             │      │             │             │            │             │
│ roberta-bas │ clarity │ smart_resam │ tone_analys │ ce   │      0.5504 │      0.6452 │     0.6494 │      0.6114 │
│ e           │         │ pling       │ is          │      │             │             │            │             │
│ distilbert- │ clarity │ none        │ none        │ ce   │      0.5454 │      0.6358 │     0.6429 │      0.6067 │
│ base-uncase │         │             │             │      │             │             │            │             │
│ d           │         │             │             │      │             │             │            │             │
│ distilbert- │ clarity │ smart_resam │ tone_analys │ ce   │      0.5101 │      0.6011 │     0.6006 │      0.5482 │
│ base-uncase │         │ pling       │ is          │      │             │             │            │             │
│ d           │         │             │             │      │             │             │            │             │
│ roberta-bas │ evasion │ smart_resam │ tone_analys │ ce   │      0.3078 │      0.2852 │     0.3084 │      0.3616 │
│ e           │         │ pling       │ is          │      │             │             │            │             │
│ distilbert- │ evasion │ smart_resam │ tone_analys │ ce   │      0.2918 │      0.2171 │     0.2370 │      0.3446 │
│ base-uncase │         │ pling       │ is          │      │             │             │            │             │
│ d           │         │             │             │      │             │             │            │             │
│ roberta-bas │ evasion │ none        │ none        │ ce   │      0.2329 │      0.2579 │     0.3084 │      0.3672 │
│ e           │         │             │             │      │             │             │            │             │
│ distilbert- │ evasion │ none        │ none        │ ce   │      0.2114 │      0.1927 │     0.2597 │      0.3044 │
│ base-uncase │         │             │             │      │             │             │            │             │
│ d           │         │             │             │      │             │             │            │             │
└─────────────┴─────────┴─────────────┴─────────────┴──────┴─────────────┴─────────────┴────────────┴─────────────┘